# Experiment 1 — Gaussian mixtures: paper figures

This notebook only loads `results/exp1/results.parquet` and plots — it never recomputes algorithm results (see CLAUDE.md and `docs/experiments.md` §0.4). Run `python -m scripts.run_experiment1` (or `SMOKE_TEST=1 python -m scripts.run_experiment1` first) to generate it.

**REVISION (2026-10-03):** this notebook was rebuilt from scratch, replacing the old many-figure/many-table version. It produces exactly two figures, both the same budget-vs-convergence layout (one subfigure per swept parameter Δ/d/N, each with a paired budget/convergence-rate plot):
1. **Main-text figure:** `kabc`, `kabc-sg`, `acb`, `acb-gaussian` — the base-method comparison. No `kabc-2x`/`kabc-0.5x` (calibration-constant ablation) and no `unif-kkm` (dropped from the roster entirely, not informative for this experiment).
2. **Appendix figure:** `kabc`, `kabc-2x`, `kabc-0.5x` — identical layout, restricted to KABC and its two calibration-constant ablations (2x / 0.5x the calibrated `c_cal`).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kabc.plotting import set_style, style_for, savefig, ALGO_ORDER, ALGO_COLORS, ALGO_LABELS

set_style()

path = Path("../results/exp1/results.parquet")
if not path.exists():
    raise FileNotFoundError(
        f"{path} not found. Run `python -m scripts.run_experiment1` (or with SMOKE_TEST=1) "
        "to generate it -- this notebook only plots existing results, it never recomputes them."
    )
df = pd.read_parquet(path)
print(f"loaded {len(df)} rows from {path}")

MAIN_ALGOS = [a for a in ["kabc", "kabc-sg", "acb", "acb-gaussian"] if a in df["algorithm"].unique()]
assert len(MAIN_ALGOS) == 4, f"expected 4 main-text algorithm rows, got {len(MAIN_ALGOS)}: {MAIN_ALGOS}"
ABLATION_ALGOS = [a for a in ["kabc", "kabc-2x", "kabc-0.5x"] if a in df["algorithm"].unique()]
assert len(ABLATION_ALGOS) == 3, f"expected 3 ablation algorithm rows, got {len(ABLATION_ALGOS)}: {ABLATION_ALGOS}"

PARAMS = [("Delta", r"$\Delta$"), ("d", "$d$"), ("N", "$N$")]
GROUP_TITLES = [r"Mean separation $\Delta$ varies", r"Dimension $d$ varies", r"Number of arms $N$ vary"]

df.head()

In [ ]:
def _set_plain_xticks(ax, values):
    values = sorted(set(values))
    ax.set_xticks(values)
    ax.set_xticklabels([
        str(int(v)) if float(v).is_integer() else str(v)
        for v in values
    ])
    ax.minorticks_off()  # drop the auto log minor ticks so they don't reappear with their own labels

def budget_vs_param_plot(swept_param, xlabel, ax, algos, overlay_theory=True):
    sub = df[df["swept_param"] == swept_param]
    for algo in algos:
        g = (sub[sub["algorithm"] == algo].groupby("swept_value")["tau"]
             .agg(["mean", "std"]).reset_index().sort_values("swept_value"))
        ax.errorbar(g["swept_value"], g["mean"], yerr=g["std"], capsize=2, **style_for(algo))
    if overlay_theory:
        theory = (sub[sub["algorithm"].isin(algos)].groupby("swept_value")["budget_cap"]
                  .mean().reset_index().sort_values("swept_value"))
        ax.plot(theory["swept_value"], theory["budget_cap"], linestyle="--", color="grey",
                linewidth=1.2, label="Budget cap")
    if swept_param == "d":
        ax.set_xscale("log")
    ax.set_yscale("log")
    _set_plain_xticks(ax, sub["swept_value"].unique())
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"mean budget ($\tau$) ± std")

def convergence_vs_param_plot(swept_param, xlabel, ax, algos):
    sub = df[df["swept_param"] == swept_param]
    for algo in algos:
        g = (sub[sub["algorithm"] == algo].groupby("swept_value")["stopped"]
             .agg(["mean", "std"]).reset_index().sort_values("swept_value"))
        ax.errorbar(g["swept_value"], g["mean"], yerr=g["std"], capsize=2, **style_for(algo))
    if swept_param == "d":
        ax.set_xscale("log")
    ax.set_ylim(-0.05, 1.05)
    _set_plain_xticks(ax, sub["swept_value"].unique())
    ax.set_xlabel(xlabel)
    ax.set_ylabel("convergence rate ± std")

def budget_convergence_figure(algos, fig_name):
    fig = plt.figure(figsize=(10, 2), layout="constrained")
    subfigs = fig.subfigures(1, 3, wspace=0.05)

    for r, (swept_param, xlabel) in enumerate(PARAMS):
        subfigs[r].suptitle(GROUP_TITLES[r], fontsize=10, fontweight="bold")
        pair_axes = subfigs[r].subplots(1, 2)
        budget_vs_param_plot(swept_param, xlabel, pair_axes[0], algos, overlay_theory=True)
        convergence_vs_param_plot(swept_param, xlabel, pair_axes[1], algos)

    handles, labels = subfigs[0].get_axes()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(algos) + 1,
               bbox_to_anchor=(0.5, -0.15), frameon=False, fontsize=8)

    savefig(fig, fig_name, fig_dir="../results/figures")
    plt.show()

## Main-text figure: kabc / kabc-sg / acb / acb-gaussian

In [ ]:
budget_convergence_figure(MAIN_ALGOS, "exp1_main")

## Appendix figure: kabc vs. its calibration-constant ablations (kabc-2x, kabc-0.5x)

Identical layout to the main-text figure above, restricted to `kabc` and its two calibration-constant variants: `kabc-2x` (2× the calibrated `c_cal`, a more conservative threshold) and `kabc-0.5x` (0.5× `c_cal`, less conservative).

In [ ]:
budget_convergence_figure(ABLATION_ALGOS, "exp1_ablation")